In [4]:
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
import random
import os

# Download necessary NLTK data
nltk.download("stopwords")
stop = set(stopwords.words("english"))

# Define file paths
papers_file = "/content/papers.csv"
paper_authors_file = "/content/paper_authors.csv"
authors_file = "/content/authors.csv"

# Download files if they don't exist
# Corrected URLs based on the repository structure

papers = pd.read_csv(papers_file, on_bad_lines='skip', engine='python', nrows=100)
paper_authors = pd.read_csv(paper_authors_file)
authors = pd.read_csv(authors_file)

def preprocess(t):
    t = str(t).lower().split()
    return [w for w in t if w.isalpha() and w not in stop]

docs_text = papers["paper_text"].astype(str).tolist()
corpus = [preprocess(t) for t in docs_text]

# Build vocabulary
vocab = {}
def wid(w):
    if w not in vocab:
        vocab[w] = len(vocab)
    return vocab[w]

docs = [[wid(w) for w in doc] for doc in corpus]
V = len(vocab)

# Map papers to authors
paper_to_authors = paper_authors.groupby("paper_id")["author_id"].apply(list).to_dict()

doc_authors = []
for pid in papers["id"]:
    doc_authors.append(paper_to_authors.get(pid, []))

# Build author index
all_authors = sorted(set(a for lst in doc_authors for a in lst))
aid = {a:i for i,a in enumerate(all_authors)}
A = len(aid)

# Model parameters
K = 100
alpha = 0.1
beta = 0.01
ITER = 10 # Iterations for LDA
ITER_AT = 10 # Iterations for AT

# Initialize LDA model parameters
nw = np.zeros((K, V))
nd = np.zeros((len(docs), K))
nk = np.zeros(K)
z = []

# LDA Initialization
for d, doc in enumerate(docs):
    topics = []
    for w in doc:
        k = random.randrange(K)
        topics.append(k)
        nw[k][w] += 1
        nd[d][k] += 1
        nk[k] += 1
    z.append(topics)

# LDA Sampling
print("Starting LDA Training...")
for it in range(ITER):
    if (it + 1) % 10 == 0:
        print("LDA Iter", it + 1)
    for d, doc in enumerate(docs):
        for i, w in enumerate(doc):
            k = z[d][i]

            nw[k][w] -= 1
            nd[d][k] -= 1
            nk[k] -= 1

            p = (nd[d] + alpha) * (nw[:, w] + beta) / (nk + V * beta)
            if p.sum() == 0: # Handle cases where all probabilities are zero
                p = np.ones_like(p) / len(p)
            else:
                p /= p.sum()

            k = np.random.choice(K, p=p)

            z[d][i] = k
            nw[k][w] += 1
            nd[d][k] += 1
            nk[k] += 1


# Initialize AT model parameters
nat = np.zeros((A, K))       # author-topic counts
nwt = np.zeros((K, V))       # topic-word counts
nt   = np.zeros(K)            # topic totals
z_at = []

# AT Initialization
print("Starting AT Training...")
for d, doc in enumerate(docs):
    authors_d_ids = doc_authors[d] # Get original author IDs for the document
    topics_d  = []
    if not authors_d_ids: # Skip documents without authors for AT model initialization
        z_at.append([])
        continue
    for w in doc:
        original_author_id = random.choice(authors_d_ids)
        mapped_author_idx = aid[original_author_id]
        t = random.randrange(K)

        topics_d.append((original_author_id, t)) # Store original author ID with topic

        nat[mapped_author_idx][t] += 1
        nwt[t][w] += 1
        nt[t] += 1
    z_at.append(topics_d)

# AT Gibbs sampling
for it in range(ITER_AT):
    if (it + 1) % 10 == 0:
        print("AT Iter", it + 1)
    for d, doc in enumerate(docs):
        authors_d_ids = doc_authors[d] # Get original author IDs for the document
        if not authors_d_ids: # Skip documents without authors for AT model sampling
            continue
        for i, w in enumerate(doc):
            if i >= len(z_at[d]): # Skip if z_at[d] is empty due to no authors
                continue

            original_author_id, t = z_at[d][i]
            mapped_author_idx = aid[original_author_id]

            nat[mapped_author_idx][t] -= 1
            nwt[t][w] -= 1
            nt[t] -= 1

            # Compute P(author, topic)
            probs = []
            for current_original_author_id in authors_d_ids:
                current_mapped_author_idx = aid[current_original_author_id]
                p = (nat[current_mapped_author_idx] + alpha) * (nwt[:,w] + beta) / (nt + V*beta)
                probs.extend(p)
            probs = np.array(probs)
            if probs.sum() == 0: # Handle cases where all probabilities are zero
                probs = np.ones_like(probs) / len(probs)
            else:
                probs /= probs.sum()

            choice = np.random.choice(len(probs), p=probs)

            a_new_original_id = authors_d_ids[choice // K] # Get the new original author ID
            t_new = choice % K
            mapped_author_idx_new = aid[a_new_original_id]

            z_at[d][i] = (a_new_original_id, t_new) # Store original author ID with topic

            nat[mapped_author_idx_new][t_new] += 1
            nwt[t_new][w] += 1
            nt[t_new] += 1

# Define calculate_perplexity function
def calculate_perplexity(nw, nd, nk, K, V, docs, alpha, beta):
    log_likelihood = 0
    total_words = 0

    # Precompute topic-word probabilities
    phi = (nw + beta) / (nk[:, None] + V * beta)

    for d, doc in enumerate(docs):
        if len(doc) == 0:
            continue

        total_words += len(doc)

        # Document topic distribution
        theta_d = (nd[d] + alpha) / (nd[d].sum() + K * alpha)

        for w in doc:
            p = np.dot(theta_d, phi[:, w])  # sum_k theta_d[k] * phi[k][w]
            log_likelihood += np.log(p + 1e-100)

    return np.exp(-log_likelihood / total_words)


# Calculate LDA Perplexity
lda_perplexity = calculate_perplexity(nw, nd, nk, K, V, docs, alpha, beta)
print(f"LDA Perplexity: {lda_perplexity}")

# Calculate AT Perplexity
nd_at = np.zeros((len(docs), K))
for d, doc_topics in enumerate(z_at):
    # Ensure doc_topics is not empty before iterating
    if doc_topics:
        for author, topic in doc_topics:
            nd_at[d, topic] += 1

at_perplexity = calculate_perplexity(nwt, nd_at, nt, K, V, docs, alpha, beta)
print(f"AT Perplexity: {at_perplexity}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Starting LDA Training...
LDA Iter 10
Starting AT Training...
AT Iter 10
LDA Perplexity: 1304.5998338046256
AT Perplexity: 1360.3701498366968
